In [2]:
import os
import sys
from hashlib import sha256
from pathlib import Path
from time import perf_counter

import mlflow
import mlflow.sklearn
import pandas as pd

from mlflow import MlflowClient
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.svm import SVC

directorio = Path.cwd().resolve()
raiz = next(
    (
        ruta for ruta in [directorio, *directorio.parents]
        if (ruta / "src" / "pipeline.py").is_file()
    ),
    None,
)

if raiz is None:
    raise FileNotFoundError("Abre 05_seleccion.ipynb desde el repositorio pset3.")

sys.path.insert(0, str(raiz))

from src.datos import cargar_obt
from src.features import construir_features
from src.pipeline import crear_pipeline

SEMILLA = 42
HORIZONTE = 5
CUANTIL_TARGET = 0.80
inicio_prueba = pd.Timestamp("2023-07-11")
capital_inicial = 10_000.0

mlflow.set_tracking_uri("http://127.0.0.1:5000")
experimento = mlflow.set_experiment(
    "Sistema de alerta temprana de volatilidad para acciones"
)
cliente = MlflowClient()

print("Raíz:", raiz)
print("URI de MLflow:", mlflow.get_tracking_uri())
print("Experimento:", experimento.name)

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


Raíz: C:\USFQ\Maestría\Ciencia_Datos\pset3
URI de MLflow: http://127.0.0.1:5000
Experimento: Sistema de alerta temprana de volatilidad para acciones


In [ ]:
df = cargar_obt()
snapshot = df.sort_values(["TRADE_DATE", "SYMBOL"])

datos_corrida = {
    "tabla_origen": "PSET2_DB.OBT.AAPL_DAILY_OBT",
    "fecha_corte_snapshot": os.environ["DATA_CUTOFF"],
    "sha256_snapshot_obt": sha256(
        snapshot.to_csv(index=False, lineterminator="\n").encode("utf-8")
    ).hexdigest(),
    "filas_snapshot": len(snapshot),
}

print("Dimensiones de la OBT:", df.shape)
print(datos_corrida)

In [ ]:
features = construir_features(df)

vol_futura_5 = (
    features["RET_LAG_0"]
    .rolling(HORIZONTE, min_periods=HORIZONTE)
    .std(ddof=1)
    .shift(-HORIZONTE)
    .rename("VOL_FUTURA_5")
)

fin_ventana_target = features.index.to_series().shift(-HORIZONTE)

datos_modelado = (
    features.iloc[60:]
    .join(vol_futura_5)
    .dropna(subset=["VOL_FUTURA_5"])
)

entrenamiento = datos_modelado.loc[
    fin_ventana_target.reindex(datos_modelado.index) < inicio_prueba
].copy()

fechas_holdout = datos_modelado.index[datos_modelado.index >= inicio_prueba]

X_desarrollo = entrenamiento.drop(columns="VOL_FUTURA_5")
vol_desarrollo = entrenamiento["VOL_FUTURA_5"]
columnas_numericas = X_desarrollo.columns.drop("MES").tolist()
cv_temporal = TimeSeriesSplit(n_splits=5, gap=HORIZONTE)

hash_desarrollo = sha256(
    entrenamiento.to_csv(index=True, lineterminator="\n").encode("utf-8")
).hexdigest()

# Retorno de t a t+1, expresado como proporción y alineado con t.
retorno_siguiente = features["RET_LAG_0"].shift(-1).div(100)

assert fin_ventana_target.loc[X_desarrollo.index].max() < fechas_holdout.min()

print("Desarrollo:", X_desarrollo.shape)
print("Fechas reservadas para prueba:", len(fechas_holdout))

In [ ]:
from math import isclose

NOMBRE_REGISTRO = "alerta-volatilidad-aapl-modelo"
uri = f"models:/{NOMBRE_REGISTRO}@champion"

version = cliente.get_model_version_by_alias(NOMBRE_REGISTRO, "champion")
run_final = cliente.get_run(version.tags["run_id_final"])

assert run_final.data.params["sha256_snapshot_obt"] == datos_corrida["sha256_snapshot_obt"]
assert run_final.data.params["sha256_dataset_desarrollo"] == hash_desarrollo
assert len(fechas_holdout) == int(run_final.data.params["filas_holdout"])
assert fechas_holdout[0].date().isoformat() == run_final.data.params["inicio_holdout"]
assert fechas_holdout[-1].date().isoformat() == run_final.data.params["fin_holdout"]

champion = mlflow.sklearn.load_model(uri)

X_prueba = datos_modelado.loc[fechas_holdout, X_desarrollo.columns]
p80 = float(run_final.data.params["p80_final"])
y_prueba = datos_modelado.loc[fechas_holdout, "VOL_FUTURA_5"].gt(p80).astype(int)

auc_recalculado = roc_auc_score(y_prueba, champion.predict_proba(X_prueba)[:, 1])
auc_registrado = run_final.data.metrics["roc_auc_test"]

assert isclose(auc_recalculado, auc_registrado, rel_tol=0, abs_tol=1e-12)

print("Champion:", uri)
print("Versión:", version.version)
print("Run ID final:", run_final.info.run_id)
print(f"ROC-AUC registrado:  {auc_registrado:.12f}")
print(f"ROC-AUC recalculado: {auc_recalculado:.12f}")
print("Verificación exitosa")

In [ ]:
carpeta_docs = raiz / "docs"
carpeta_docs.mkdir(exist_ok=True)

corridas = mlflow.search_runs(experiment_ids=[experimento.experiment_id])
ruta_csv = carpeta_docs / "corridas.csv"
corridas.to_csv(ruta_csv, index=False)

print(f"{len(corridas)} corridas exportadas a {ruta_csv}")